# ML Assignment 2 – Credit Risk Classification
**Name:** Roshni K  |  **BITS ID:** 2026AK05661

Dataset: UCI Statlog (German Credit Data) – 1000 applicants, 20 features, target: good (0) / bad (1) credit risk.



In [2]:
print("BITS ID: 2026AK05661 - ML Assignment 2 - executed on BITS Virtual Lab")
import platform, datetime; print("Python", platform.python_version(), "|", datetime.datetime.now())

BITS ID: 2026AK05661 - ML Assignment 2 - executed on BITS Virtual Lab
Python 3.12.7 | 2026-11-08 00:14:17.559250


In [3]:
%pip install --upgrade numpy scipy scikit-learn pandas jolib

Note: you may need to restart the kernel to use updated packages.


In [4]:
# Install libraries (only needed once)
%pip install -q scikit-learn pandas numpy joblib

Note: you may need to restart the kernel to use updated packages.


## 1. Imports

In [6]:
import os
import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score, matthews_corrcoef,
                             precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

print("scikit-learn version:", sklearn.__version__)
RANDOM_STATE = 42
TARGET = "credit_risk"   # 1 = bad credit risk, 0 = good
os.makedirs("model", exist_ok=True)

scikit-learn version: 1.9.1


## 2. Load the dataset and decode the UCI category codes

In [8]:
COLUMNS = [
    "checking_status", "duration_months", "credit_history", "purpose", "credit_amount",
    "savings", "employment_since", "installment_rate", "personal_status_sex", "other_debtors",
    "residence_since", "property", "age", "other_installment_plans", "housing",
    "existing_credits", "job", "num_dependents", "own_telephone", "foreign_worker", "class",
]
CODES = {
    "checking_status": {"A11": "< 0 DM", "A12": "0-200 DM", "A13": ">= 200 DM", "A14": "no checking account"},
    "credit_history": {"A30": "no credits / all paid", "A31": "all paid at this bank",
                       "A32": "existing paid duly", "A33": "delay in past", "A34": "critical account"},
    "purpose": {"A40": "car (new)", "A41": "car (used)", "A42": "furniture/equipment",
                "A43": "radio/television", "A44": "domestic appliances", "A45": "repairs",
                "A46": "education", "A47": "vacation", "A48": "retraining",
                "A49": "business", "A410": "others"},
    "savings": {"A61": "< 100 DM", "A62": "100-500 DM", "A63": "500-1000 DM",
                "A64": ">= 1000 DM", "A65": "unknown / none"},
    "employment_since": {"A71": "unemployed", "A72": "< 1 year", "A73": "1-4 years",
                         "A74": "4-7 years", "A75": ">= 7 years"},
    "personal_status_sex": {"A91": "male: divorced/separated", "A92": "female: div/sep/married",
                            "A93": "male: single", "A94": "male: married/widowed", "A95": "female: single"},
    "other_debtors": {"A101": "none", "A102": "co-applicant", "A103": "guarantor"},
    "property": {"A121": "real estate", "A122": "savings agreement / life insurance",
                 "A123": "car or other", "A124": "unknown / none"},
    "other_installment_plans": {"A141": "bank", "A142": "stores", "A143": "none"},
    "housing": {"A151": "rent", "A152": "own", "A153": "for free"},
    "job": {"A171": "unemployed / unskilled non-resident", "A172": "unskilled resident",
            "A173": "skilled employee", "A174": "management / highly qualified"},
    "own_telephone": {"A191": "none", "A192": "yes"},
    "foreign_worker": {"A201": "yes", "A202": "no"},
}
CATEGORICAL = list(CODES.keys())
NUMERIC = ["duration_months", "credit_amount", "installment_rate", "residence_since",
           "age", "existing_credits", "num_dependents"]

# works whether german_raw.csv is next to the notebook or inside a data/ folder
path = "german_raw.csv" if os.path.exists("german_raw.csv") else "data/german_raw.csv"
df = pd.read_csv(path, header=None, names=COLUMNS)
for col, mapping in CODES.items():
    df[col] = df[col].map(mapping)
df[TARGET] = (df.pop("class") == 2).astype(int)

print(f"Dataset: {df.shape[0]} rows, {df.shape[1] - 1} features "
      f"({len(NUMERIC)} numeric, {len(CATEGORICAL)} categorical)")
print("Class balance:", df[TARGET].value_counts().to_dict(), "| Missing values:", df.isna().sum().sum())
df.head()

Dataset: 1000 rows, 20 features (7 numeric, 13 categorical)
Class balance: {0: 700, 1: 300} | Missing values: 0


,checking_status,duration_months,credit_history,purpose,credit_amount,savings,employment_since,installment_rate,personal_status_sex,other_debtors,...,property,age,other_installment_plans,housing,existing_credits,job,num_dependents,own_telephone,foreign_worker,credit_risk
0,< 0 DM,6,critical account,radio/television,1169,unknown / none,>= 7 years,4,male: single,none,...,real estate,67,none,own,2,skilled employee,1,yes,yes,0
1,0-200 DM,48,existing paid duly,radio/television,5951,< 100 DM,1-4 years,2,female: div/sep/married,none,...,real estate,22,none,own,1,skilled employee,1,none,yes,1
2,no checking account,12,critical account,education,2096,< 100 DM,4-7 years,2,male: single,none,...,real estate,49,none,own,1,unskilled resident,2,none,yes,0
3,< 0 DM,42,existing paid duly,furniture/equipment,7882,< 100 DM,4-7 years,2,male: single,guarantor,...,savings agreement / life insurance,45,none,for free,1,skilled employee,2,none,yes,0
4,< 0 DM,24,delay in past,car (new),4870,< 100 DM,1-4 years,3,male: single,none,...,unknown / none,53,none,for free,2,skilled employee,2,none,yes,1


## 3. Train / test split (80 / 20, stratified) and save test data

In [10]:
X, y = df.drop(columns=[TARGET]), df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

test_df = X_test.copy(); test_df[TARGET] = y_test
test_df.to_csv("test_data.csv", index=False)
print("Train:", X_train.shape, "| Test:", X_test.shape, "-> saved test_data.csv")

Train: (800, 20) | Test: (200, 20) -> saved test_data.csv


## 4. Pre-processing: scale numeric features, one-hot encode categorical features

In [12]:
def preprocessor():
    return ColumnTransformer([
        ("num", StandardScaler(), NUMERIC),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL),
    ])

## 5. Define the 5 models

In [14]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced",
                                              random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(max_depth=3, min_samples_leaf=10,
                                            class_weight="balanced", random_state=RANDOM_STATE),
    "KNN": KNeighborsClassifier(n_neighbors=7),
    "Naive Bayes": GaussianNB(),
    "Support Vector Machine": SVC(kernel="rbf", class_weight="balanced", random_state=RANDOM_STATE),
}

## 6. Train, evaluate (6 metrics) and save each model

In [16]:
def get_scores(model, X):
    """Probabilities for AUC if available, otherwise SVM decision values."""
    if hasattr(model, "predict_proba"):
        try:
            return model.predict_proba(X)[:, 1]
        except AttributeError:
            pass
    return model.decision_function(X)

rows = []
for name, clf in models.items():
    pipe = Pipeline([("prep", preprocessor()), ("clf", clf)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    rows.append({
        "ML Model Name": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "AUC": roc_auc_score(y_test, get_scores(pipe, X_test)),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "MCC": matthews_corrcoef(y_test, y_pred),
    })
    fname = "model/" + name.lower().replace(" ", "_") + ".pkl"
    joblib.dump(pipe, fname)
    print(f"{name:<24} confusion matrix [TN FP FN TP] = {confusion_matrix(y_test, y_pred).ravel()}  -> saved {fname}")

joblib.dump(list(X.columns), "model/feature_names.pkl")

Logistic Regression      confusion matrix [TN FP FN TP] = [102  38  12  48]  -> saved model/logistic_regression.pkl
Decision Tree            confusion matrix [TN FP FN TP] = [75 65  9 51]  -> saved model/decision_tree.pkl
KNN                      confusion matrix [TN FP FN TP] = [126  14  39  21]  -> saved model/knn.pkl
Naive Bayes              confusion matrix [TN FP FN TP] = [100  40  21  39]  -> saved model/naive_bayes.pkl
Support Vector Machine   confusion matrix [TN FP FN TP] = [106  34  14  46]  -> saved model/support_vector_machine.pkl


['model/feature_names.pkl']

## 7. Comparison table

In [18]:
results = pd.DataFrame(rows).set_index("ML Model Name").round(4)
results.to_csv("model/metrics.csv")
results

,Accuracy,AUC,Precision,Recall,F1,MCC
ML Model Name,,,,,,
Logistic Regression,0.750,0.8058,0.5581,0.8000,0.6575,0.4893
Decision Tree,0.630,0.6959,0.4397,0.8500,0.5795,0.3581
KNN,0.735,0.7357,0.6000,0.3500,0.4421,0.3015
Naive Bayes,0.695,0.7411,0.4937,0.6500,0.5612,0.3415
Support Vector Machine,0.760,0.8032,0.5750,0.7667,0.6571,0.4900
